<a href="https://colab.research.google.com/github/awaisusa005-cell/stabblediffusion/blob/main/stable/stable_diffusion_webui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess
from typing import List, Optional, Tuple, Union

# ── Shared paths ──────────────────────────────────────────────────────────────
WEBUI_ROOT = "/content/stable-diffusion-webui"
EXTENSIONS_DIR = f"{WEBUI_ROOT}/extensions"
CONTROLNET_MODELS_DIR = f"{EXTENSIONS_DIR}/sd-webui-controlnet/models"

# ── Shared aria2c flags ───────────────────────────────────────────────────────
ARIA2C_FLAGS = "--console-log-level=error -c -x 16 -s 16 -k 1M"


def run(cmd: str) -> None:
    """Run a shell command, raising on failure."""
    print(f">>> {cmd}")
    subprocess.check_call(cmd, shell=True)


def clone_repo(url: str, dest: str, branch: Optional[str] = None) -> None:
    """Clone a git repository to *dest*, optionally checking out *branch*."""
    branch_flag = f"-b {branch} " if branch else ""
    run(f"git clone {branch_flag}{url} {dest}")


def clone_extension(url: str, name: Optional[str] = None) -> None:
    """Clone a git repo into the WebUI extensions directory."""
    name = name or url.rstrip("/").split("/")[-1]
    clone_repo(url, f"{EXTENSIONS_DIR}/{name}")


def install_extensions(repos: List[Union[str, Tuple[str, str]]]) -> None:
    """Clone multiple extensions.

    Each entry is either a plain URL (directory name is derived from the URL)
    or a ``(url, name)`` tuple when an explicit directory name is needed.
    """
    for entry in repos:
        if isinstance(entry, tuple):
            clone_extension(entry[0], entry[1])
        else:
            clone_extension(entry)


def aria2c_download(url: str, dest_dir: str, filename: Optional[str] = None) -> None:
    """Download a single file with aria2c using the shared flags."""
    filename = filename or url.split("/")[-1]
    run(f"aria2c {ARIA2C_FLAGS} {url} -d {dest_dir} -o {filename}")


def batch_aria2c_download(
    filenames: List[str],
    base_url: str,
    dest_dir: str,
) -> None:
    """Download a batch of files that share the same base URL and destination."""
    for f in filenames:
        aria2c_download(f"{base_url}/{f}", dest_dir)


print("Shared utilities loaded.")

In [ ]:
# ── System setup & dependencies ───────────────────────────────────────────────
%cd /content

%env TF_CPP_MIN_LOG_LEVEL=1

!apt -y update -qq
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

!apt -y install -qq aria2 libcairo2-dev pkg-config python3-dev
!pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 torchtext==0.15.2 torchdata==0.6.1 --extra-index-url https://download.pytorch.org/whl/cu118 -U
!pip install -q xformers==0.0.20 triton==2.0.0 gradio_client==0.2.7 -U

In [ ]:
# ── Clone WebUI and core resources ────────────────────────────────────────────
clone_repo("https://github.com/camenduru/stable-diffusion-webui", WEBUI_ROOT, branch="v2.4")
clone_repo("https://huggingface.co/embed/negative", f"{WEBUI_ROOT}/embeddings/negative")
clone_repo("https://huggingface.co/embed/lora", f"{WEBUI_ROOT}/models/Lora/positive")

aria2c_download(
    "https://huggingface.co/embed/upscale/resolve/main/4x-UltraSharp.pth",
    f"{WEBUI_ROOT}/models/ESRGAN",
)
run(
    f"wget https://raw.githubusercontent.com/camenduru/stable-diffusion-webui-scripts/main/run_n_times.py "
    f"-O {WEBUI_ROOT}/scripts/run_n_times.py"
)

In [ ]:
# ── Install extensions (data-driven) ──────────────────────────────────────────
EXTENSION_REPOS = [
    "https://github.com/deforum-art/deforum-for-automatic1111-webui",
    "https://github.com/camenduru/stable-diffusion-webui-images-browser",
    "https://github.com/camenduru/stable-diffusion-webui-huggingface",
    "https://github.com/camenduru/sd-civitai-browser",
    "https://github.com/kohya-ss/sd-webui-additional-networks",
    "https://github.com/Mikubill/sd-webui-controlnet",
    "https://github.com/fkunn1326/openpose-editor",
    "https://github.com/jexom/sd-webui-depth-lib",
    "https://github.com/hnmr293/posex",
    "https://github.com/nonnonstop/sd-webui-3d-open-pose-editor",
    "https://github.com/camenduru/sd-webui-tunnels",
    "https://github.com/etherealxx/batchlinks-webui",
    "https://github.com/camenduru/stable-diffusion-webui-catppuccin",
    "https://github.com/AUTOMATIC1111/stable-diffusion-webui-rembg",
    "https://github.com/ashen-sensored/stable-diffusion-webui-two-shot",
    "https://github.com/thomasasfk/sd-webui-aspect-ratio-helper",
    "https://github.com/tjm35/asymmetric-tiling-sd-webui",
]

install_extensions(EXTENSION_REPOS)

%cd {WEBUI_ROOT}
!git reset --hard
!git -C {WEBUI_ROOT}/repositories/stable-diffusion-stability-ai reset --hard

In [ ]:
# ── Download ControlNet v1.1 models & configs ────────────────────────────────
CONTROLNET_BASE = "https://huggingface.co/ckpt/ControlNet-v1-1"

CONTROLNET_SAFETENSORS = [
    "control_v11e_sd15_ip2p_fp16.safetensors",
    "control_v11e_sd15_shuffle_fp16.safetensors",
    "control_v11p_sd15_canny_fp16.safetensors",
    "control_v11f1p_sd15_depth_fp16.safetensors",
    "control_v11p_sd15_inpaint_fp16.safetensors",
    "control_v11p_sd15_lineart_fp16.safetensors",
    "control_v11p_sd15_mlsd_fp16.safetensors",
    "control_v11p_sd15_normalbae_fp16.safetensors",
    "control_v11p_sd15_openpose_fp16.safetensors",
    "control_v11p_sd15_scribble_fp16.safetensors",
    "control_v11p_sd15_seg_fp16.safetensors",
    "control_v11p_sd15_softedge_fp16.safetensors",
    "control_v11p_sd15s2_lineart_anime_fp16.safetensors",
    "control_v11f1e_sd15_tile_fp16.safetensors",
]

CONTROLNET_YAMLS = [f.replace(".safetensors", ".yaml") for f in CONTROLNET_SAFETENSORS]

T2I_ADAPTERS = [
    "t2iadapter_style_sd14v1.pth",
    "t2iadapter_sketch_sd14v1.pth",
    "t2iadapter_seg_sd14v1.pth",
    "t2iadapter_openpose_sd14v1.pth",
    "t2iadapter_keypose_sd14v1.pth",
    "t2iadapter_depth_sd14v1.pth",
    "t2iadapter_color_sd14v1.pth",
    "t2iadapter_canny_sd14v1.pth",
    "t2iadapter_canny_sd15v2.pth",
    "t2iadapter_depth_sd15v2.pth",
    "t2iadapter_sketch_sd15v2.pth",
    "t2iadapter_zoedepth_sd15v1.pth",
]

batch_aria2c_download(CONTROLNET_SAFETENSORS, f"{CONTROLNET_BASE}/resolve/main", CONTROLNET_MODELS_DIR)
batch_aria2c_download(CONTROLNET_YAMLS,       f"{CONTROLNET_BASE}/raw/main",     CONTROLNET_MODELS_DIR)
batch_aria2c_download(T2I_ADAPTERS,           f"{CONTROLNET_BASE}/resolve/main", CONTROLNET_MODELS_DIR)

# SD v1.4 checkpoint
aria2c_download(
    "https://huggingface.co/ckpt/sd14/resolve/main/sd-v1-4.ckpt",
    f"{WEBUI_ROOT}/models/Stable-diffusion",
)

In [ ]:
# ── Patches & launch ──────────────────────────────────────────────────────────
!sed -i -e '''/from modules import launch_utils/a\import os''' {WEBUI_ROOT}/launch.py
!sed -i -e '''/        prepare_environment()/a\        os.system\(f\"""sed -i -e ''\"s/dict()))/dict())).cuda()/g\"'' {WEBUI_ROOT}/repositories/stable-diffusion-stability-ai/ldm/util.py""")''' {WEBUI_ROOT}/launch.py
!sed -i -e 's/\["sd_model_checkpoint"\]/\["sd_model_checkpoint","sd_vae","CLIP_stop_at_last_layers"\]/g' {WEBUI_ROOT}/modules/shared.py

!python launch.py --listen --xformers --enable-insecure-extension-access --theme dark --gradio-queue --multiple